In [1]:
import torch
import os
import torch.nn.functional as F

# Directory containing .pt files
directory = '/export/c09/lavanya/languageIdentification/test_embedding_40'

# List all .pt files in the directory
pt_files = [f for f in os.listdir(directory) if f.endswith('.pt')]

/home/lshankar1/miniconda3/envs/capstone/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
for pt_file in pt_files:
    file_path = os.path.join(directory, pt_file)
    data = torch.load(file_path)
    embeddings = data['reshaped_embeddings']
    for i, audio_embedding in enumerate(embeddings):
        print(f"File: {pt_file}")
        print(f"Layer {i} embedding shape: {audio_embedding.shape}")
        print(f"shape of embedding first element: {audio_embedding.shape[0]}")
        print(f"shape of embedding second element: {audio_embedding.shape[1]}")
        print(f"shape of embedding third element: {audio_embedding[0]}")

File: emb_TTS_P91182TT_VCST_ECxxx_01_AO_48503281_v001_R004_CRR_MERLIon-CCS_segment_0.pt
Layer 0 embedding shape: torch.Size([1, 315, 384])
shape of embedding first element: 1
shape of embedding second element: 315
shape of embedding third element: tensor([[ 0.0403,  0.0779, -0.0678,  ..., -0.0184,  0.0422, -0.0362],
        [ 0.0684,  0.0143, -0.0615,  ...,  0.0753, -0.0295, -0.1025],
        [ 0.0366,  0.0655,  0.0375,  ..., -0.0224,  0.0346, -0.1194],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]])
File: emb_TTS_P91182TT_VCST_ECxxx_01_AO_48503281_v001_R004_CRR_MERLIon-CCS_segment_0.pt
Layer 1 embedding shape: torch.Size([1, 315, 384])
shape of embedding first element: 1
shape of embedding second element: 315
shape of embedding third element: tensor([[ 0.0817, -0.0243,  0.0633,  ...,  0.0032,  0.1101,  0.0113],


In [4]:
import pandas as pd
import librosa
import soundfile as SF
import pickle
with open('/export/c09/lavanya/languageIdentification/test_pickles_40/updated_segments_with_embeddings.pkl', 'rb') as f:
    seg_df = pickle.load(f)

In [5]:
seg_df.head()

,segmented_audio,embedding,language_tag,overlap_diff_lang,length,utt_id
0,/export/c09/lavanya/languageIdentification/tes...,/export/c09/lavanya/languageIdentification/tes...,English,False,1580,a1
1,/export/c09/lavanya/languageIdentification/tes...,/export/c09/lavanya/languageIdentification/tes...,English,False,3190,a4
2,/export/c09/lavanya/languageIdentification/tes...,/export/c09/lavanya/languageIdentification/tes...,English,False,1020,a8
3,/export/c09/lavanya/languageIdentification/tes...,/export/c09/lavanya/languageIdentification/tes...,English,False,1160,a11
4,/export/c09/lavanya/languageIdentification/tes...,/export/c09/lavanya/languageIdentification/tes...,English,False,3480,a14


In [41]:
print("unique languages in the dataset: ", seg_df['language_tag'].unique())

unique languages in the dataset:  ['English']


In [7]:
seg_df['embedding'][1]

'/export/c09/lavanya/languageIdentification/test_embedding_40/emb_TTS_P91182TT_VCST_ECxxx_01_AO_48503281_v001_R004_CRR_MERLIon-CCS_segment_1.pt'

In [8]:
seg_df['language_tag'][1]

'English'

In [44]:
import torch
import pandas as pd

# Function to load embeddings from a file path
def load_embedding(file_path):
    data = torch.load(file_path)
    return data['reshaped_embeddings']

# Prepare lists for embeddings and labels
embeddings_list = []
labels_list = []

# Iterate over the DataFrame rows
for _, row in seg_df.iterrows():
    file_path = row['embedding']
    #print(file_path)
    label = row['language_tag']
    #print(label)
    
    # Load the embedding from the file
    if os.path.exists(file_path):
        embeddings = load_embedding(file_path)
        # print("embeddings",embeddings)
        # print("shape of embeddings",len(embeddings))
        # print("shape of embeddings[0]",embeddings[0])
        # print("shape of embeddings",len(embeddings[0]))
        # Assuming we want to use embeddings from the first layer
        layer_embeddings = embeddings[0].squeeze(0)  # Remove singleton dimension
        print("layer_embeddings",layer_embeddings)
        # print("shape of layer_embeddings",layer_embeddings.shape)
        # Collect embeddings and labels
        embeddings_list.append(layer_embeddings)
        labels_list.append(label)
    else:
        print(f"File {file_path} does not exist.")

# Convert lists to tensors
X = torch.stack(embeddings_list)
y = torch.tensor([0 if label == 'English' else 1 for label in labels_list])  # Example: binary classification

X

layer_embeddings tensor([[ 0.0403,  0.0779, -0.0678,  ..., -0.0184,  0.0422, -0.0362],
        [ 0.0684,  0.0143, -0.0615,  ...,  0.0753, -0.0295, -0.1025],
        [ 0.0366,  0.0655,  0.0375,  ..., -0.0224,  0.0346, -0.1194],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]])
layer_embeddings tensor([[-0.0332,  0.0001,  0.0651,  ..., -0.0745, -0.0454,  0.0325],
        [-0.0524, -0.0244,  0.0220,  ..., -0.0246,  0.0119,  0.0130],
        [-0.0071, -0.0821,  0.0223,  ...,  0.0140,  0.0198,  0.0458],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]])
layer_embeddings tensor([[ 0.0258,  0.1284, -0.0760,  ...,  0.0056,  0.0310, -0.0047],
        [-0

tensor([[[ 4.0315e-02,  7.7893e-02, -6.7827e-02,  ..., -1.8428e-02,
           4.2237e-02, -3.6245e-02],
         [ 6.8382e-02,  1.4290e-02, -6.1509e-02,  ...,  7.5293e-02,
          -2.9523e-02, -1.0252e-01],
         [ 3.6647e-02,  6.5466e-02,  3.7495e-02,  ..., -2.2392e-02,
           3.4604e-02, -1.1938e-01],
         ...,
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
           0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
           0.0000e+00,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
           0.0000e+00,  0.0000e+00]],

        [[-3.3209e-02,  1.4539e-04,  6.5122e-02,  ..., -7.4526e-02,
          -4.5393e-02,  3.2545e-02],
         [-5.2371e-02, -2.4361e-02,  2.1991e-02,  ..., -2.4614e-02,
           1.1933e-02,  1.3021e-02],
         [-7.1110e-03, -8.2100e-02,  2.2330e-02,  ...,  1.4029e-02,
           1.9821e-02,  4.5766e-02],
         ...,
         [ 0.0000e+00,  0

In [46]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

# Define the model
class SimpleNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = x.mean(dim=1)  # Pooling across time steps
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Load embeddings
def load_embedding(file_path):
    data = torch.load(file_path)
    return data['reshaped_embeddings']

# Prepare data
def prepare_data(df):
    embeddings_list = []
    labels_list = []
    
    for _, row in df.iterrows():
        file_path = row['embedding']
        label = row['language_tag']
        
        if os.path.exists(file_path):
            embeddings = load_embedding(file_path)
            layer_embeddings = embeddings[3].squeeze(0)  # Remove singleton dimension
            embeddings_list.append(layer_embeddings)
            labels_list.append(label)
        else:
            print(f"File {file_path} does not exist.")
    
    X = torch.stack(embeddings_list)
    y = torch.tensor([0 if label == 'English' else 1 for label in labels_list])  # Example: binary classification
    return X, y

# Load data
df = pd.read_pickle('/export/c09/lavanya/languageIdentification/test_pickles_40/updated_segments_with_embeddings.pkl')
X, y = prepare_data(df)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define hyperparameters
input_dim = 384
hidden_dim = 128
output_dim = 2
batch_size = 32
num_epochs = 10
learning_rate = 0.001

# Initialize model, loss function, and optimizer
model = SimpleNN(input_dim, hidden_dim, output_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Create DataLoader
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Training loop
for epoch in range(num_epochs):
    model.train()
    for embeddings_batch, labels_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(embeddings_batch)
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()
    
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}')

# Save the model
torch.save(model.state_dict(), 'model.pth')

# Evaluation function
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for embeddings_batch, labels_batch in loader:
            outputs = model(embeddings_batch)
            _, predicted = torch.max(outputs, 1)
            total += labels_batch.size(0)
            correct += (predicted == labels_batch).sum().item()
    accuracy = 100 * correct / total
    return accuracy

# Evaluate the model
train_accuracy = evaluate(model, train_loader)
test_accuracy = evaluate(model, test_loader)

print(f'Train Accuracy: {train_accuracy:.2f}%')
print(f'Test Accuracy: {test_accuracy:.2f}%')


Epoch 1/10, Loss: 0.6797020435333252
Epoch 2/10, Loss: 0.6729671955108643
Epoch 3/10, Loss: 0.6662659645080566
Epoch 4/10, Loss: 0.6596297025680542
Epoch 5/10, Loss: 0.6530705690383911
Epoch 6/10, Loss: 0.6465848088264465
Epoch 7/10, Loss: 0.6401581168174744
Epoch 8/10, Loss: 0.6337786316871643
Epoch 9/10, Loss: 0.627440333366394
Epoch 10/10, Loss: 0.6211244463920593
Train Accuracy: 100.00%
Test Accuracy: 100.00%


In [ ]:

Epoch 1/10, Loss: 0.6585243940353394
Epoch 2/10, Loss: 0.6412218809127808
Epoch 3/10, Loss: 0.630264401435852
Epoch 4/10, Loss: 0.6112244129180908
Epoch 5/10, Loss: 0.6043533086776733
Epoch 6/10, Loss: 0.5787158608436584
Epoch 7/10, Loss: 0.5583139061927795
Epoch 8/10, Loss: 0.5440343022346497
Epoch 9/10, Loss: 0.515868067741394
Epoch 10/10, Loss: 0.503436803817749

Epoch 1/10, Loss: 0.6774011850357056
Epoch 2/10, Loss: 0.6642423272132874
Epoch 3/10, Loss: 0.6484318971633911
Epoch 4/10, Loss: 0.6311360597610474
Epoch 5/10, Loss: 0.6237879991531372
Epoch 6/10, Loss: 0.6088020205497742
Epoch 7/10, Loss: 0.594538152217865
Epoch 8/10, Loss: 0.5823306441307068
Epoch 9/10, Loss: 0.5681266188621521
Epoch 10/10, Loss: 0.5481239557266235


Epoch 1/10, Loss: 0.7433914542198181
Epoch 2/10, Loss: 0.7309573888778687
Epoch 3/10, Loss: 0.7134871482849121
Epoch 4/10, Loss: 0.7039785385131836
Epoch 5/10, Loss: 0.6911501884460449
Epoch 6/10, Loss: 0.6795985102653503
Epoch 7/10, Loss: 0.6694208979606628
Epoch 8/10, Loss: 0.6535201072692871
Epoch 9/10, Loss: 0.6341577768325806
Epoch 10/10, Loss: 0.6238186359405518


Epoch 1/10, Loss: 0.6467949748039246
Epoch 2/10, Loss: 0.6307516098022461
Epoch 3/10, Loss: 0.6191290616989136
Epoch 4/10, Loss: 0.6020818948745728
Epoch 5/10, Loss: 0.5804850459098816
Epoch 6/10, Loss: 0.5830661058425903
Epoch 7/10, Loss: 0.5516340732574463
Epoch 8/10, Loss: 0.5498618483543396
Epoch 9/10, Loss: 0.5396772623062134
Epoch 10/10, Loss: 0.5167513489723206

Epoch 1/10, Loss: 0.7154065370559692
Epoch 2/10, Loss: 0.7001928091049194
Epoch 3/10, Loss: 0.6866888999938965
Epoch 4/10, Loss: 0.6711708307266235
Epoch 5/10, Loss: 0.6543787717819214
Epoch 6/10, Loss: 0.6518778800964355
Epoch 7/10, Loss: 0.6388301849365234
Epoch 8/10, Loss: 0.6185181736946106
Epoch 9/10, Loss: 0.6057355999946594
Epoch 10/10, Loss: 0.5950506925582886

